<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
# Desafio 4, LSTM Bot QA

### Datos
El objetivo es utilizar datos disponibles del challenge ConvAI2 (Conversational Intelligence Challenge 2) de conversaciones en inglés. Se construirá un BOT para responder a preguntas del usuario (QA).\
[LINK](http://convai.io/data/)

In [4]:
!pip install --upgrade --no-cache-dir gdown --quiet

In [5]:
import re
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer, one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Activation, Dropout, Dense, Flatten, LSTM, SimpleRNN, Embedding, Input
from sklearn.model_selection import train_test_split

In [6]:
# Descargar la carpeta de dataset
import os
import gdown
if os.access('data_volunteers.json', os.F_OK) is False:
    url = 'https://drive.google.com/uc?id=1awUxYwImF84MIT5-jCaYAPe2QwSgS1hN&export=download'
    output = 'data_volunteers.json'
    gdown.download(url, output, quiet=False)
else:
    print("El dataset ya se encuentra descargado")

Downloading...
From: https://drive.google.com/uc?id=1awUxYwImF84MIT5-jCaYAPe2QwSgS1hN
To: /content/data_volunteers.json
100%|██████████| 2.58M/2.58M [00:00<00:00, 78.4MB/s]


In [7]:
# dataset_file
import json

text_file = "data_volunteers.json"
with open(text_file) as f:
    data = json.load(f) # la variable data será un diccionario



In [8]:
# Observar los campos disponibles en cada linea del dataset
data[0].keys()

dict_keys(['dialog', 'start_time', 'end_time', 'bot_profile', 'user_profile', 'eval_score', 'profile_match', 'participant1_id', 'participant2_id'])

In [9]:
chat_in = []
chat_out = []

input_sentences = []
output_sentences = []
output_sentences_inputs = []
max_len = 80 # Subimos el límite para tener más datos

def clean_text(txt):
    txt = txt.lower()
    # Guardamos los cambios asignando a la variable
    txt = txt.replace("\'d", " had")
    txt = txt.replace("\'s", " is")
    txt = txt.replace("\'m", " am")
    txt = txt.replace("don't", " do not")
    # Quitamos caracteres especiales pero mantenemos espacios
    txt = re.sub(r'[^a-z0-0\s]', '', txt)
    return txt.strip()

for line in data:
    for i in range(len(line['dialog'])-1):
        chat_in = clean_text(line['dialog'][i]['text'])
        chat_out = clean_text(line['dialog'][i+1]['text'])

        # Filtramos solo si son extremadamente largas o vacías
        if len(chat_in) < 1 or len(chat_out) < 1 or len(chat_in) > max_len:
            continue

        input_sentences.append(chat_in)
        output_sentences.append(chat_out + ' <eos>')
        output_sentences_inputs.append('<sos> ' + chat_out)

print("Cantidad de rows utilizadas ahora:", len(input_sentences))

Cantidad de rows utilizadas ahora: 13094


In [10]:
input_sentences[1], output_sentences[1], output_sentences_inputs[1]

('hi how are you', 'not bad and you <eos>', '<sos> not bad and you')

### ***Pueden realizar el desafio en Keras o PyTorch.***

### 2 - Preprocesamiento
Realizar el preprocesamiento necesario para obtener:
- word2idx_inputs, max_input_len
- word2idx_outputs, max_out_len, num_words_output
- encoder_input_sequences, decoder_output_sequences, decoder_targets

In [11]:
# Re-definimos los tokens especiales
SOS = '<sos>'
EOS = '<eos>'

input_sentences = []
output_sentences = [] # El target (lo que queremos que prediga)
output_sentences_inputs = [] # Lo que entra al decoder (teacher forcing)

for line in data:
    for i in range(len(line['dialog'])-1):
        chat_in = clean_text(line['dialog'][i]['text'])
        chat_out = clean_text(line['dialog'][i+1]['text'])

        if len(chat_in) < 2 or len(chat_out) < 2: # Filtro mínimo
            continue

        input_sentences.append(chat_in)
        output_sentences.append(chat_out + ' ' + EOS)
        output_sentences_inputs.append(SOS + ' ' + chat_out)

# Tokenización de entradas
tokenizer_in = Tokenizer(filters='')
tokenizer_in.fit_on_texts(input_sentences)
encoder_input_sequences = tokenizer_in.texts_to_sequences(input_sentences)
word2idx_inputs = tokenizer_in.word_index
max_input_len = max(len(s) for s in encoder_input_sequences)

# Tokenización de salidas
tokenizer_out = Tokenizer(filters='')
tokenizer_out.fit_on_texts(output_sentences + output_sentences_inputs)
decoder_input_sequences = tokenizer_out.texts_to_sequences(output_sentences_inputs)
decoder_target_sequences = tokenizer_out.texts_to_sequences(output_sentences)
word2idx_outputs = tokenizer_out.word_index
max_out_len = max(len(s) for s in decoder_target_sequences)
num_words_output = len(word2idx_outputs) + 1

# Padding
encoder_input_sequences = pad_sequences(encoder_input_sequences, maxlen=max_input_len, padding='post')
decoder_input_sequences = pad_sequences(decoder_input_sequences, maxlen=max_out_len, padding='post')
decoder_target_sequences = pad_sequences(decoder_target_sequences, maxlen=max_out_len, padding='post')

### 3 - Preparar los embeddings
Utilizar los embeddings de Glove o FastText para transformar los tokens de entrada en vectores

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# Descarga de Glove (si no lo tenés en el entorno)
# !wget https://nlp.stanford.edu/data/glove.6B.zip
# !unzip glove.6B.zip

embeddings_index = {}
with open('/content/drive/MyDrive/glove.6B.300d.txt', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

# Crear la matriz para el Encoder
num_words_in = len(word2idx_inputs) + 1
embedding_matrix_in = np.zeros((num_words_in, 300))
for word, i in word2idx_inputs.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix_in[i] = embedding_vector

### 4 - Entrenar el modelo
Entrenar un modelo basado en el esquema encoder-decoder utilizando los datos generados en los puntos anteriores. Utilce como referencias los ejemplos vistos en clase.

In [16]:
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model

latent_dim = 512

# --- ENCODER ---
encoder_inputs = Input(shape=(max_input_len,))
enc_emb = Embedding(num_words_in, 300, weights=[embedding_matrix_in], trainable=False)(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_state=True, dropout=0.2)

encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]


# --- DECODER ---
decoder_inputs = Input(shape=(max_out_len,))
dec_emb_layer = Embedding(num_words_output, 100)
dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, dropout=0.2)

decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(num_words_output, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Modelo para entrenamiento
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(
    [encoder_input_sequences, decoder_input_sequences],
    decoder_target_sequences,
    batch_size=128,
    epochs=20
)

Epoch 1/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 58s 491ms/step - accuracy: 0.9653 - loss: 0.5519
Epoch 2/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 58s 557ms/step - accuracy: 0.9777 - loss: 0.1341
Epoch 3/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 57s 551ms/step - accuracy: 0.9783 - loss: 0.1261
Epoch 4/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 58s 561ms/step - accuracy: 0.9788 - loss: 0.1228
Epoch 5/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 58s 559ms/step - accuracy: 0.9794 - loss: 0.1184
Epoch 6/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 58s 562ms/step - accuracy: 0.9803 - loss: 0.1138
Epoch 7/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 58s 560ms/step - accuracy: 0.9811 - loss: 0.1082
Epoch 8/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 58s 561ms/step - accuracy: 0.9819 - loss: 0.1029
Epoch 9/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 58s 560ms/step - accuracy: 0.9825 - loss: 0.0986
Epoch 10/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 59s 564ms/step - accuracy: 0.9831 - loss: 0.0949
Epoch 11/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 58s 562ms/step - accuracy: 0.9835 - loss: 0.0916
Epoch 12/20
104/104

### 5 - Inferencia
Experimentar el funcionamiento de su modelo. Recuerde que debe realizar la inferencia de los modelos por separado de encoder y decoder.

In [19]:
idx2word_inputs = {v: k for k, v in word2idx_inputs.items()}
idx2word_outputs = {v: k for k, v in word2idx_outputs.items()}

# 1. Modelo del Encoder
encoder_model = Model(encoder_inputs, encoder_states)

# 2. Modelo del Decoder (Inferencia)
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_emb_inf = dec_emb_layer(decoder_inputs)
decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(dec_emb_inf, initial_state=decoder_states_inputs)
decoder_states_inf = [state_h_inf, state_c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = Model([decoder_inputs] + decoder_states_inputs, [decoder_outputs_inf] + decoder_states_inf)

def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq, verbose=0)
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = word2idx_outputs['<sos>']

    stop_condition = False
    decoded_sentence = ''
    temperature = 0.6  # <--- Probá entre 0.4 y 0.8. Más alto = más creativo/loco.

    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)

        # Obtenemos las probabilidades del último token
        probs = output_tokens[0, -1, :]

        # Aplicamos Temperatura (esto suaviza las probabilidades)
        probs = np.asarray(probs).astype('float64')
        probs = np.log(probs + 1e-10) / temperature
        exp_preds = np.exp(probs)
        probs = exp_preds / np.sum(exp_preds)

        # Elegimos una palabra al azar basándonos en esa nueva distribución
        sampled_token_index = np.random.choice(len(probs), p=probs)

        # Evitamos que elija el token 0 (padding)
        if sampled_token_index == 0:
            continue

        sampled_word = idx2word_outputs.get(sampled_token_index, '')

        if sampled_word == '<eos>' or len(decoded_sentence.split()) > max_out_len:
            stop_condition = True
        else:
            decoded_sentence += ' ' + sampled_word

        target_seq[0, 0] = sampled_token_index
        states_value = [h, c]

    return decoded_sentence

In [20]:
def chat_with_bot(text):
    # Limpiamos y tokenizamos la entrada del usuario
    text_cleaned = clean_text(text)
    input_seq = tokenizer_in.texts_to_sequences([text_cleaned])
    input_seq = pad_sequences(input_seq, maxlen=max_input_len, padding='post')

    # Generamos la respuesta
    response = decode_sequence(input_seq)

    print(f"User: {text}")
    print(f"Bot: {response.strip()}")
    print("-" * 30)

# --- E. PRUEBAS FINALES ---
# Ahora podés hablarle al bot:
chat_with_bot("hello")
chat_with_bot("how are you")
chat_with_bot("do you have any pets")

User: hello
Bot: what do you do for a living
------------------------------
User: how are you
Bot: oh that is a good idea
------------------------------
User: do you have any pets
Bot: i am doing well what are you
------------------------------


# Conclusiones


Al implementar Temperature Sampling ($\tau = 0.6$), se logró mitigar el problema del determinismo extremo en la inferencia. El modelo pasó de generar una única respuesta dominante ('i am a student') a producir secuencias variadas. Esto demuestra que el conocimiento semántico estaba presente en los pesos de la red de 512 unidades latentes, pero se veía limitado por el uso de Greedy Search (argmax)